In [1]:
from typing import Tuple, List

import os
import rootutils

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [2]:
from src.utils import create_df, compute_fingerprints, compute_descriptors, create_data, eval_metrics, plot_pred_true, plot_importance

In [3]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from argparse import Namespace

from sklearn.model_selection import train_test_split

---
# Feature Extraction:

In [4]:
descriptors_all = [
    "MolWt",
    "LogP",
    "TPSA",
    "NumRotatableBonds",
    "NumHDonors",
    "NumHAcceptors",
    "FractionCSP3",
    "NumAromaticRings",
    "FractionRotatableBonds",
    "CalcNumAmideBonds",
    "rdMolDescriptors.CalcNumBridgeheadAtoms",
    "rdMolDescriptors.CalcNumSaturatedRings",
    "fr_Al_COO",
    "fr_Al_OH",
    "fr_Al_OH_noTert",
    "fr_ArN",
    "fr_Ar_COO",
    "fr_Ar_N",
    "fr_Ar_NH",
    "fr_Ar_OH",
    "fr_COO",
    "fr_C_O",
    "fr_C_O_noCOO",
    "fr_C_S",
    "fr_HOCCN",
    "fr_Imine",
    "fr_NH0",
    "fr_NH1",
    "fr_NH2",
    "fr_N_O",
    "fr_Ndealkylation1",
    "fr_Ndealkylation2",
    "fr_Nhpyrrole",
    "fr_SH",
    "fr_aldehyde",
    "fr_alkyl_carbamate",
    "fr_alkyl_halide",
    "fr_allylic_oxid",
    "fr_amide",
    "fr_amidine",
    "fr_aniline",
    "fr_aryl_methyl",
    "fr_azide",
    "fr_azo",
    "fr_barbitur",
    "fr_benzene",
    "fr_benzodiazepine",
    "fr_bicyclic",
    "fr_diazo",
    "fr_dihydropyridine",
    "fr_epoxide",
    "fr_ester",
    "fr_ether",
    "fr_furan",
    "fr_guanido",
    "fr_halogen",
    "fr_hdrzine",
    "fr_hdrzone",
    "fr_imidazole",
    "fr_imide",
    "fr_isocyan",
    "fr_isothiocyan",
    "fr_ketone",
    "fr_ketone_Topliss",
    "fr_lactam",
    "fr_lactone",
    "fr_methoxy",
    "fr_morpholine",
    "fr_nitrile",
    "fr_nitro",
    "fr_nitro_arom_nonortho",
    "fr_nitroso",
    "fr_oxazole",
    "fr_oxime",
    "fr_para_hydroxylation",
    "fr_phenol",
    "fr_phenol_noOrthoHbond",
    "fr_phos_acid",
    "fr_phos_ester",
    "fr_piperdine",
    "fr_piperzine",
    "fr_priamide",
    # "rdMolDescriptors.CalcEccentricity",
    # "rdMolDescriptors.CalcPBF",
    # "rdMolDescriptors.CalcSpherocityIndex",
    # "rdMolDescriptors.CalcRadiusOfGyration",
    "NumHBD",
    "NumHeavyAtoms",
    "NumHBA",
    "NumRings",
    "NumHeteroatoms",
    "Chi0v",
    "Chi1v",
    "Chi2v",
    "Chi3v",
    "Chi4v"
]

descriptors_short = [
    "NumHBD",                # Number of Hydrogen Bond Donors (rdMolDescriptors)
    "NumHeavyAtoms",         # Number of Heavy Atoms

    'MolWt',                 # Molecular Weight
    'LogP',                  # LogP (octanol-water partition coefficient)
    'TPSA',                  # Topological Polar Surface Area
    'NumRotatableBonds',     # Number of Rotatable Bonds
    'NumHDonors',            # Number of Hydrogen Bond Donors
    'NumHAcceptors',         # Number of Hydrogen Bond Acceptors
    'FractionCSP3',          # Fraction of sp3 Hybridized Carbons
    'NumAromaticRings',      # Number of Aromatic Rings
    'FractionRotatableBonds',# Fraction of Rotatable Bonds
    'NumHBD',               
    'NumHBA',                # Number of Hydrogen Bond Acceptors (rdMolDescriptors)
    'NumRings',              # Number of Rings
    'NumHeteroatoms',        # Number of Heteroatoms
    'Chi0v',                 # Chi Connectivity Index 0 (Valence)
    'Chi1v',                 # Chi Connectivity Index 1 (Valence)
    'Chi2v',                 # Chi Connectivity Index 2 (Valence)
]

In [5]:
data_path = "data/Bradley_dataset_ok_3"
columns = ['line_number', 'smiles', 'cas', 'label', 'T']

df = create_df(data_path, columns)

In [6]:
data_args = {
    "descriptors": descriptors_all,

    "apply_norm": True,
    
    "create_fingerprints": False,
    "temp_column": True
}

In [7]:
df = df[:100]

_, X, labels, temp = create_data(df, **data_args)

100%|██████████| 100/100 [00:01<00:00, 88.67it/s]


---
# MultiTask Learning with Lightning:

In [8]:
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning

import torchmetrics
from lightning import LightningModule, Trainer, seed_everything
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from torchmetrics.classification import Accuracy, F1Score, AUROC
from torchmetrics import R2Score, MeanSquaredError, MeanAbsoluteError

from torch.utils.data import DataLoader, TensorDataset

In [32]:

class MultiTaskModel(LightningModule):
    def __init__(self, input_dim, hidden_dim=256, drop=0.2):
        super().__init__()
        self.save_hyperparameters()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(hidden_dim, 1)
        self.regressor = nn.Linear(hidden_dim, 1)

        self.val_accuracy = Accuracy(task='binary')
        self.val_f1_macro = F1Score(task='binary', average='macro')
        self.val_f1_weighted = F1Score(task='binary', average='weighted')
        self.val_roc_auc = AUROC(task='binary')
        self.val_r2_class = R2Score()
        self.val_mse = MeanSquaredError()
        self.val_mae = MeanAbsoluteError()
        self.val_r2_reg = R2Score()

    def forward(self, x):
        shared_out = self.shared(x)
        class_logits = self.classifier(shared_out)
        reg_output = self.regressor(shared_out)
        return class_logits, reg_output

    def training_step(self, batch, batch_idx):
        x, labels, temp = batch
        logits, predictions = self(x)
        classification_loss = F.binary_cross_entropy_with_logits(logits, labels.unsqueeze(1).float())
        regression_loss = F.mse_loss(predictions, temp.unsqueeze(1).float())
        loss = classification_loss + regression_loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, labels, temp = batch
        logits, predictions = self(x)
        classification_loss = F.binary_cross_entropy_with_logits(logits, labels.unsqueeze(1).float())
        regression_loss = F.mse_loss(predictions, temp.unsqueeze(1).float())
        loss = classification_loss + regression_loss
        self.log("val_loss", loss, prog_bar=True)

        probs = torch.sigmoid(logits).squeeze()
        preds = (probs > 0.5).long()
        labels_long = labels.long()

        self.val_accuracy(preds, labels_long)
        self.val_f1_macro(preds, labels_long)
        self.val_f1_weighted(preds, labels_long)
        self.val_roc_auc(probs, labels_long)
        self.val_r2_class(probs, labels.float())
        self.val_mse(predictions.squeeze(), temp)
        self.val_mae(predictions.squeeze(), temp)
        self.val_r2_reg(predictions.squeeze(), temp)

        return loss

    def on_validation_epoch_end(self):
        # Classification:
        self.log("val_acc", self.val_accuracy.compute())
        self.log("val_f1_macro", self.val_f1_macro.compute())
        self.log("val_f1_weighted", self.val_f1_weighted.compute())
        self.log("val_roc_auc", self.val_roc_auc.compute())
        self.log("val_r2_class", self.val_r2_class.compute())
        # Regression:
        self.log("val_mse", self.val_mse.compute())
        self.log("val_mae", self.val_mae.compute())
        self.log("val_r2", self.val_r2_reg.compute())

        self.val_accuracy.reset()
        self.val_f1_macro.reset()
        self.val_f1_weighted.reset()
        self.val_roc_auc.reset()
        self.val_r2_class.reset()
        
        self.val_mse.reset()
        self.val_mae.reset()
        self.val_r2_reg.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [33]:
seed_everything(42, verbose=False)

def create_datasets(X, labels, temp, train_size=0.8) -> Tuple[TensorDataset]:
    dataset_components = train_test_split(X, labels, temp, train_size=train_size)

    # np.ndarray -> torch.tensor:
    X_train, X_val, y_train, y_val, temp_train, temp_val = list(map(lambda x: torch.tensor(x, dtype=torch.float32), dataset_components))
    
    train_data = TensorDataset(X_train, y_train, temp_train)
    val_data = TensorDataset(X_val, y_val, temp_val)

    return train_data, val_data

In [34]:
cfg = Namespace(
    project_name="bradley",
    
    train_size=0.8,

    batch_size=32,
    lr=1e-3,
    max_epochs=10,

    hid_dim=256,
    drop=0.2,
)

In [35]:
train_data, val_data = create_datasets(X, labels, temp, train_size=cfg.train_size)

train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False)

In [40]:
input_dim = X.shape[1]
model = MultiTaskModel(
    input_dim=input_dim
)

current_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

logger = TensorBoardLogger(
    save_dir="tb_logs/", name=cfg.project_name
)

checkpoint_callback = ModelCheckpoint(
    dirpath=f"checkpoints/{current_date}-{cfg.project_name}",
    filename="{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    monitor="val_loss",
    mode="min",
    save_last=True,
)

trainer = Trainer(
    logger=logger,
    callbacks=[checkpoint_callback],
    max_epochs=cfg.max_epochs,
)

AttributeError: module 'datetime' has no attribute 'now'

In [39]:

trainer.fit(model, train_loader, val_loader)



   | Name            | Type              | Params | Mode 
---------------------------------------------------------------
0  | shared          | Sequential        | 89.6 K | train
1  | classifier      | Linear            | 257    | train
2  | regressor       | Linear            | 257    | train
3  | val_accuracy    | BinaryAccuracy    | 0      | train
4  | val_f1_macro    | BinaryF1Score     | 0      | train
5  | val_f1_weighted | BinaryF1Score     | 0      | train
6  | val_roc_auc     | BinaryAUROC       | 0      | train
7  | val_r2_class    | R2Score           | 0      | train
8  | val_mse         | MeanSquaredError  | 0      | train
9  | val_mae         | MeanAbsoluteError | 0      | train
10 | val_r2_reg      | R2Score           | 0      | train
---------------------------------------------------------------
90.1 K    Trainable params
0         Non-trainable params
90.1 K    Total params
0.360     Total estimated model params size (MB)
16        Modules in train mode
0         Mod

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
